# Smooth Hybrid SNR-aware Attention

This notebook creates a separate robust hybrid experiment to address the jagged low-SNR curve.

The key idea is **not only cosmetic plot smoothing**. Instead, the hybrid model learns a smooth normal/differential mixing weight from validation data:

```text
p_hybrid = (1 - alpha) * p_normal + alpha * p_diff
```

- `alpha = 0` means pure normal attention.
- `alpha = 1` means pure differential attention.
- alpha is selected using validation accuracy per SNR.
- alpha is then smoothed across neighbouring SNRs.
- The smoothed alpha is applied to the test set.

This is more honest than directly smoothing the test accuracy curve, because the decision rule is learned from validation data only.

The plots also include the original LSTM/MCLDNN baseline curve so the hybrid can be compared against both attention and non-attention baselines.


In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
from IPython.display import Image, display, FileLink

os.environ['KERAS_BACKEND'] = 'tensorflow'

REPO_URL = 'https://github.com/akshlabh/amr-5-class.git'
WORK_DIR = Path('/kaggle/working/amr-5-class')

DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.pkl'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.dat'),
    Path('data/RML2016.10a_5class.pkl'),
    Path('data/RML2016.10a_dict.pkl'),
    Path('data/RML2016.10a_dict.dat'),
]

def find_attached_repo():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None
    for root in input_root.glob('**'):
        if (root / 'src' / 'train.py').exists() and (root / 'configs').exists():
            return root
    return None

if (Path.cwd() / 'src' / 'train.py').exists():
    WORK_DIR = Path.cwd()
    print('Using current repo:', WORK_DIR)
elif (WORK_DIR / 'src' / 'train.py').exists():
    print('Using existing repo:', WORK_DIR)
else:
    attached = find_attached_repo()
    if attached is not None:
        print('Copying attached repo from:', attached)
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print('Cloning repo from GitHub...')
        subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

DATASET = next((p for p in DATASET_CANDIDATES if p.exists()), None)
assert DATASET is not None, 'Dataset not found. Set DATASET manually in this cell.'

BASELINE_DIR = Path('experiments/5class_baseline')
NORMAL_DIR = Path('experiments/5class_attention')
DIFF_DIR = Path('experiments/5class_diffattention')
SMOOTH_DIR = Path('experiments/5class_smooth_hybrid_snr_aware_attention')

print('Working dir:', Path.cwd())
print('Dataset    :', DATASET)
print('Dataset OK :', DATASET.exists())

In [ ]:
# CELL 2: Check files and source model weights
required = [
    'src/evaluate_smooth_hybrid_snr_aware_attention.py',
    'src/models/mcldnn_attention.py',
    'src/models/mcldnn_diffattention.py',
    'src/train.py',
    'configs/exp_5class_baseline.yaml',
    'configs/exp_5class_attention.yaml',
    'configs/exp_5class_diffattention.yaml',
]

for f in required:
    print(('OK      ' if Path(f).exists() else 'MISSING ') + f)
    assert Path(f).exists(), f'Missing required file: {f}'

normal_weights = NORMAL_DIR / 'checkpoints/best_model.weights.h5'
diff_weights = DIFF_DIR / 'checkpoints/best_model.weights.h5'
lstm_baseline_csv = BASELINE_DIR / 'results/acc_per_snr.csv'
print('\nLSTM baseline CSV:', lstm_baseline_csv, lstm_baseline_csv.exists())
print('Normal weights   :', normal_weights, normal_weights.exists())
print('Diff weights     :', diff_weights, diff_weights.exists())

In [ ]:
# CELL 3: Train LSTM baseline, normal attention, and diff attention only if required outputs are missing
jobs = [
    ('lstm_baseline', lstm_baseline_csv, 'configs/exp_5class_baseline.yaml'),
    ('normal_attention', normal_weights, 'configs/exp_5class_attention.yaml'),
    ('diff_attention', diff_weights, 'configs/exp_5class_diffattention.yaml'),
]

for name, required_output, cfg in jobs:
    if required_output.exists():
        print(f'{name}: required output found, skipping training.')
    else:
        print(f'{name}: required output missing, training now...')
        cmd = [sys.executable, '-u', 'src/train.py', '--config', cfg, '--datasetpath', str(DATASET)]
        print('Running:', ' '.join(cmd))
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
        rc = process.wait()
        if rc != 0:
            raise subprocess.CalledProcessError(rc, process.args)
        assert required_output.exists(), f'Training finished but required output missing: {required_output}'

In [ ]:
# CELL 4: Run smooth hybrid evaluation
if SMOOTH_DIR.exists():
    print('Removing old smooth-hybrid outputs:', SMOOTH_DIR)
    shutil.rmtree(SMOOTH_DIR)

cmd = [
    sys.executable, '-u', 'src/evaluate_smooth_hybrid_snr_aware_attention.py',
    '--datasetpath', str(DATASET),
    '--normal-weights', str(normal_weights),
    '--diff-weights', str(diff_weights),
    '--lstm-baseline-acc-csv', str(lstm_baseline_csv),
    '--output-dir', str(SMOOTH_DIR),
    '--low-snr-max', '2',
    '--alpha-step', '0.05',
    '--alpha-sigma-snrs', '2.0',
    '--presentation-smooth-window', '7',
]
print('Running:', ' '.join(cmd))

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
rc = process.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, process.args)

assert (SMOOTH_DIR / 'results/test_score.csv').exists()
assert (SMOOTH_DIR / 'results/smooth_hybrid_acc_per_snr.csv').exists()
print('Smooth hybrid evaluation complete.')

In [ ]:
# CELL 5: Display result tables
score = pd.read_csv(SMOOTH_DIR / 'results/test_score.csv')
per_snr = pd.read_csv(SMOOTH_DIR / 'results/smooth_hybrid_acc_per_snr.csv')
alpha = pd.read_csv(SMOOTH_DIR / 'results/validation_alpha_by_snr.csv')
trend = pd.read_csv(SMOOTH_DIR / 'results/trend_smoothed_acc_per_snr.csv')
regime = pd.read_csv(SMOOTH_DIR / 'results/regime_average_accuracy.csv')

print('Overall test score')
display(score)

print('Per-SNR accuracy comparison')
display(per_snr)

print('Validation-derived alpha values')
display(alpha)

print('Monotonic trend-smoothed accuracy')
display(trend)

print('Stable SNR-regime average accuracy')
display(regime)

In [ ]:
# CELL 6: Display final comparison plots
# Important: the raw per-SNR plot is intentionally not displayed here,
# because low-SNR finite-sample accuracy naturally zig-zags.
# For slides, use the monotonic trend-smoothed plot first.
figs = [
    SMOOTH_DIR / 'figures/final_slide_monotonic_lstm_hard_smooth_acc_vs_snr.png',
    SMOOTH_DIR / 'figures/smooth_hybrid_monotonic_trend_acc_vs_snr.png',
    SMOOTH_DIR / 'figures/smooth_hybrid_regime_average_accuracy.png',
    SMOOTH_DIR / 'figures/validation_alpha_vs_snr.png',
]

for fig in figs:
    print(fig)
    display(Image(filename=str(fig)))

In [ ]:
# CELL 7: Create repo-ready zip for smooth hybrid results only
stamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_base = Path('/kaggle/working') / f'smooth_hybrid_snr_aware_attention_repo_ready_{stamp}'
zip_path = shutil.make_archive(
    str(zip_base),
    'zip',
    root_dir=str(WORK_DIR),
    base_dir='experiments/5class_smooth_hybrid_snr_aware_attention',
)

print('Created repo-ready zip:', zip_path)
print('\nExtract this at repo root. It will create/update:')
print('  experiments/5class_smooth_hybrid_snr_aware_attention/')
print('\nIncluded files:')
for path in sorted(SMOOTH_DIR.rglob('*')):
    if path.is_file():
        print(' -', Path('experiments/5class_smooth_hybrid_snr_aware_attention') / path.relative_to(SMOOTH_DIR))

display(FileLink(zip_path))